# Create a Knowledge Assistant (Product Manuals)

This notebook creates a [Knowledge Assistant](https://docs.databricks.com/api/workspace/knowledgeassistants/createknowledgesource) for **cordless drill/driver manuals** via the REST API.

The assistant indexes the original PDF manuals from the volume, so users can ask detailed questions that go beyond the structured fields -- e.g. safety instructions, maintenance procedures, or warranty information.

**API:** `POST /api/2.1/knowledge-assistants`

In [ ]:
# %pip install databricks-sdk -q

from databricks.sdk import WorkspaceClient
from manage_knowledge_assistant import *

w = WorkspaceClient()

# Configuration – product manuals
DISPLAY_NAME = "power-tool-manuals-assistant"  # 4–63 chars, only \w . -
DESCRIPTION = (
    "Answers questions about cordless drill/driver manuals from Bosch, Makita, DeWalt, and Milwaukee. "
    "Can help with safety instructions, maintenance procedures, troubleshooting, warranty details, "
    "and technical specifications that go beyond the structured extraction."
)
INSTRUCTIONS = (
    "You are a power tool expert assistant. Answer questions using the product manuals provided. "
    "When comparing tools, reference specific model numbers and manufacturers. "
    "Always cite which manual the information comes from. "
    "If the manuals are in multiple languages, prefer the English sections."
)

DOCUMENTS_VOLUME_PATH = "/Volumes/data_extraction/default/documents/productmanuals"
KNOWLEDGE_SOURCE_DISPLAY_NAME = "power-tool-pdf-manuals"
KNOWLEDGE_SOURCE_DESCRIPTION = (
    "Original PDF manuals for cordless drill/drivers from Bosch, Makita, DeWalt, and Milwaukee."
)

In [ ]:
# Create Knowledge Assistant (uses config: DISPLAY_NAME, DESCRIPTION, INSTRUCTIONS)
response = create_knowledge_assistant(w, DISPLAY_NAME, DESCRIPTION, INSTRUCTIONS)

print("Knowledge Assistant created.")
print(f"id: {response.get('id')}")

In [ ]:
# Create a files Knowledge Source on the documents volume (run Create Assistant cell first)
source_response = create_knowledge_source_files(
    w,
    response.get("id"),
    display_name=KNOWLEDGE_SOURCE_DISPLAY_NAME,
    description=KNOWLEDGE_SOURCE_DESCRIPTION,
    volume_path=DOCUMENTS_VOLUME_PATH,
)
print("Knowledge Source (files) created.")
print(f"  id: {source_response.get('id')}")
print(f"  name: {source_response.get('name')}")

In [ ]:
# Get status of the created Knowledge Assistant
status = get_knowledge_assistant(w, response.get("id"))
print("Knowledge Assistant status:")
print(f"  id: {status.get('id')}")
print(f"  name: {status.get('name')}")
print(f"  display_name: {status.get('display_name')}")
print(f"  state: {status.get('state')}")  # CREATING | ACTIVE | FAILED

In [ ]:
# Get Knowledge Source status (run after Create Knowledge Source cell)
source_status = get_knowledge_source(w, response.get("id"), source_response.get("id"))
print("Knowledge Source status:")
print(f"  id: {source_status.get('id')}")
print(f"  name: {source_status.get('name')}")
print(f"  display_name: {source_status.get('display_name')}")
print(f"  state: {source_status.get('state')}")  # UPDATING | UPDATED | FAILED_UPDATE